In [ ]:
# Core imports
import os
from typing import Dict, Any, List, Optional, Literal, Annotated, TypedDict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from datetime import datetime
import operator
import json
import uuid
import requests

# LangChain imports
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper

# LangGraph imports
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langgraph.errors import NodeInterrupt

# Display imports
from IPython.display import Image, display

In [ ]:
#!!pip install langchain langchain-anthropic langgraph python-dotenv pydantic langchain_community requests

In [ ]:
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from utils.env_loader import load_environment
load_environment()
# Show which .env file is being used
env_candidates = [
    project_root / ".env",
    Path.cwd() / ".env",
    Path(".env"),
]

for p in env_candidates:
    print(f"{p} -> exists: {p.exists()}")

# If the project root .env exists, that's the one most likely used
env_path = project_root / ".env"
print(f"Likely .env used: {env_path}")

In [ ]:
load_environment()
env_file = project_root / ".env"
print(f"Using env file: {env_file}")
load_environment(str(env_file))

In [ ]:
import os

def mask_value(value: str) -> str:
    if value is None:
        return "None"
    if not value:
        return ""
    if len(value) <= 8:
        return "***"
    return f"{value[:4]}...{value[-2:]}"

# Print the resolved env file path
print(f"project_root={project_root}")
print(f"env_path={env_path}")
print(f"env_file={env_file}")
print(f"env_file_exists={env_file.exists()}")

# Print variables already loaded into the environment
print("\nLoaded environment variables:")
for key in sorted(os.environ):
    if any(token in key.upper() for token in ["KEY", "SECRET", "TOKEN", "PASSWORD", "API"]):
        print(f"{key}={mask_value(os.environ.get(key, ''))}")
    elif key in {"ENV", "APP_ENV", "PYTHONPATH"}:
        print(f"{key}={os.environ.get(key, '')}")

# If the .env file is present, print its contents (masked)
dotenv_path = env_path if env_path.exists() else env_file
print(f"\n.env file path: {dotenv_path}")
if dotenv_path.exists():
    print("Contents of .env (masked):")
    with open(dotenv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            print(f"{key.strip()}={mask_value(value.strip())}")

In [ ]:
import os
from pathlib import Path

def save_current_env_to_file(path: str | Path = None, keys: list[str] | None = None):
    if path is None:
        path = project_root / ".env"
    path = Path(path)

    # Preserve existing entries already in the file
    existing = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            existing[k.strip()] = v.strip()

    # Only persist common environment variables
    if keys is None:
        keys = [
            "ENV", "APP_ENV", "PYTHONPATH",
            "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "PINECONE_API_KEY",
            "GOOGLE_API_KEY", "SERPER_API_KEY", "TAVILY_API_KEY",
            "LANGCHAIN_API_KEY", "API_KEY", "SECRET", "TOKEN", "PASSWORD"
        ]

    # Add/update current env values
    for key, value in os.environ.items():
        upper = key.upper()
        if key in existing or any(token in upper for token in ["KEY", "SECRET", "TOKEN", "PASSWORD", "API"]) or key in {"ENV", "APP_ENV", "PYTHONPATH"}:
            if key in keys or any(token in upper for token in ["KEY", "SECRET", "TOKEN", "PASSWORD", "API"]) or key in {"ENV", "APP_ENV", "PYTHONPATH"}:
                existing[key] = str(value)

    # Write back to .env
    with path.open("w", encoding="utf-8") as f:
        for key in sorted(existing):
            f.write(f"{key}={existing[key]}\n")

    print(f"Saved environment variables to: {path}")
    print(f"Total entries: {len(existing)}")

save_current_env_to_file()

In [ ]:
import os
from pathlib import Path

required_keys = ["ANTHROPIC_API_KEY", "SERPER_API_KEY", "LINKEDIN_COOKIE"]

def ensure_env_vars_from_dotenv(dotenv_path: str | Path | None = None, keys: list[str] | None = None):
    if dotenv_path is None:
        dotenv_path = project_root / ".env"
    dotenv_path = Path(dotenv_path)
    if keys is None:
        keys = required_keys

    found = False
    if dotenv_path.exists():
        for line in dotenv_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if key in keys:
                os.environ[key] = value
                found = True
                print(f"Loaded {key} into os.environ")
    else:
        print(f"No .env file found at: {dotenv_path}")

    for key in keys:
        if key not in os.environ:
            print(f"{key} is still missing from os.environ")
        else:
            print(f"{key} is set in os.environ")

    return found

# Load the required variables explicitly
ensure_env_vars_from_dotenv(project_root / ".env", required_keys)

# Optional: if you want to force reload from a different env file
# ensure_env_vars_from_dotenv("/absolute/path/to/.env", required_keys)

# Quick check
for key in required_keys:
    print(f"{key} = {'SET' if os.environ.get(key) else 'MISSING'}")

In [ ]:
# Load the .env file from the project directory
dotenv_dir = Path("/Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents")
dotenv_path = dotenv_dir / ".env"

# Fallback to the already-known project_root if needed
if not dotenv_path.exists():
    dotenv_path = project_root / ".env"

if not dotenv_path.exists():
    raise FileNotFoundError(f"No .env file found at: {dotenv_path}")

load_environment(str(dotenv_path))
print(f"Loaded environment from: {dotenv_path}")

# Optional: verify a few expected keys
for key in ["ANTHROPIC_API_KEY", "SERPER_API_KEY", "LINKEDIN_COOKIE"]:
    print(f"{key} = {'SET' if os.environ.get(key) else 'MISSING'}")

In [ ]:
import getpass
from pathlib import Path
import os

# Use existing notebook variables if present
dotenv = Path(globals().get("dotenv_path", project_root / ".env"))
keys = globals().get("required_keys", ["ANTHROPIC_API_KEY", "SERPER_API_KEY", "LINKEDIN_COOKIE"])

missing = [k for k in keys if not os.environ.get(k)]
if not missing:
    print("All required keys are set.")
else:
    print("Missing keys:", missing)
    if dotenv.exists():
        backup = dotenv.with_suffix(dotenv.suffix + ".bak")
        backup.write_text(dotenv.read_text(encoding="utf-8"), encoding="utf-8")
        print(f"Backed up {dotenv} -> {backup}")
    for key in missing:
        val = getpass.getpass(f"Enter value for {key} (input hidden): ").strip()
        if not val:
            print(f"Skipping {key}; no value entered.")
            continue
        os.environ[key] = val
        with dotenv.open("a", encoding="utf-8") as f:
            f.write(f"{key}={val}\n")
        print(f"Set and appended {key} to {dotenv}")

In [ ]:
import sys
print(sys.executable)

In [ ]:
from dotenv import load_dotenv
load_dotenv("/Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents/.env")

In [ ]:
import os
print("Environment variables loaded from .env:")
for key in ["ANTHROPIC_API_KEY", "SERPER_API_KEY", "LINKEDIN_COOKIE"]:
    print(f"{key} = {'SET' if os.environ.get(key) else 'MISSING'}")

In [ ]:
import sys
from pathlib import Path

# Setup paths
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from utils.env_loader import load_environment
load_environment()